# 🌲 Needle 3 互動教學：專為微型邊緣端打造的自動化基底模型

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Child-pi/needle/blob/main/needle_tutorial.ipynb)

歡迎來到 **Needle 3** 的互動式實作教學！

### 🌟 什麼是 Needle 3？
傳統大語言模型 (如 GPT-4, Claude) 體積龐大 (數十至數百 GB)，依賴雲端 API、延遲高且有隱私疑慮；而傳統端側小模型又極易產生格式錯誤與胡言亂語。

**Needle 3 是專為手機、手錶、智慧家居、機器人與微控制器設計的端側基底模型**：
- 📦 **極致微型**：僅 **8 ~ 29 MB** 體積（CQ2 2.125-bit 量化），單一二進制檔直接載入。
- ⚡ **超高速度**：在樹莓派 5 (Raspberry Pi 5) 上高達 **10,000 tokens/s Prefill** 與 **4,000 tokens/s Decode**。
- 🎯 **零格式錯誤**：字節級語法約束解碼 (Byte-level Grammar)，保證 100% 符合 JSON Schema。
- 🛡️ **拒絕幻覺**：自帶校準置信度 (Calibrated Confidence) 與接地驗證，不支援的請求主動返回空列表拒絕。
- 💻 **跨平臺原生運行**：支援 13 種架構（macOS, Linux, Windows, Android, iOS, watchOS, WASM）。

本筆記本將帶您逐步體驗 Needle 3 的核心能力！

--- 
## 步驟 1：安裝 Needle 運行時環境

Needle 採用純輕量 Python 封裝，底層原生引擎小於 1 MB，不需要龐大的 PyTorch 或 JAX 依賴（僅需 1~2 秒即可安裝完成）。
初次呼叫時，引擎會自動從 Hugging Face 下載不到 35 MB 的量化權重並快取在本地。

In [ ]:
# 安裝 cactus-needle 與 pydantic
!pip install -q cactus-needle pydantic

import needle
print(f"✅ Needle 套件載入成功！版本: {needle.__version__}")

--- 
## 步驟 2：快速上手 — 定義第一個工具 (Hello World Tool Calling)

在 Needle 中，定義工具只需撰寫標準 Python 函式，並加上 `@needle.tool` 裝飾器：
- **函式簽名 (Type Annotations)**：自動決定參數型別（如 `str`, `int`, `Literal`）。
- **Docstring 說明**：自動成為模型理解該工具的描述。
- **預設值**：有預設值的參數自動標記為 Optional。
- **`agent.run(query)`**：會自動完成「意圖分析 ➔ 參數填充 ➔ 執行 Python 函式 ➔ 回傳結果」的完整閉環！

In [ ]:
import json
from typing import Literal

# 1. 定義一個查詢天氣的工具
@needle.tool
def get_weather(city: str, units: Literal["celsius", "fahrenheit"] = "celsius"):
    """查詢指定城市當前的天氣狀況與溫度。
    
    Args:
        city: 城市名稱，例如 '台北', '東京', '舊金山'
        units: 溫度單位，可選 celsius 或 fahrenheit
    """
    # 模擬硬體/網路 API 返回結果
    return {
        "city": city,
        "temperature": 25 if units == "celsius" else 77,
        "units": units,
        "condition": "晴天 (Clear Sky)"
    }

# 2. 初始化 Needle Agent
agent = needle.Needle(tools=[get_weather])

# 3. 自然語言提問並自動執行
user_query = "請問現在東京的天氣如何？請用攝氏溫度回覆。"
response = agent.run(user_query)

# 4. 檢視結構化輸出
print("=== 模型回應與執行結果 ===")
print(json.dumps(response, ensure_ascii=False, indent=2))

print("\n=== 最終工具執行傳回的結果 ===")
print(response["results"])

### 💡 輸出結構說明：
- `type`: 回應型態，`'call'` 代表產生了工具調用。
- `function_calls`: 模型按順序抽取的函式名稱與保證合法的參數字典。
- `reasoning`: 模型從自然語言中萃取各欄位的推理鏈。
- `confidence`: 模型經過校準後的置信度評分 (0.0 ~ 1.0)。
- `results`: 自動調用 Python 函式後取得的真實回傳值！

--- 
## 步驟 3：多工具協同與複合意圖 (Multi-Tool Complex Control)

在智慧家庭或機器人應用中，使用者的一句話往往包含多個硬體動作。
Needle 能在一輪推論中**按順序生成多個動作**，並透過 `needle.Field` 對數值範圍設定約束條件。

In [ ]:
from typing import Annotated, Optional

Room = Literal["kitchen", "living_room", "bedroom", "study"]

@needle.tool
def control_lights(
    room: Room,
    action: Literal["on", "off", "dim"],
    brightness_percent: Annotated[Optional[int], needle.Field(ge=0, le=100)] = None,
    color: Optional[Literal["warm white", "cool white", "blue", "red"]] = None,
):
    """控制房間燈光的開關狀態、亮度與顏色。"""
    return {"device": "light", "room": room, "action": action, "brightness": brightness_percent, "color": color}

@needle.tool
def set_thermostat(temperature: Annotated[int, needle.Field(ge=16, le=30)]):
    """設定室內恆溫空調溫度（攝氏度）。"""
    return {"device": "thermostat", "target_temp": temperature}

@needle.tool
def start_robot_vacuum(
    action: Literal["start", "stop", "dock"],
    room: Optional[Room] = None
):
    """啟動掃地機器人、派遣至指定房間清掃或返回充電座。"""
    return {"device": "vacuum", "action": action, "target_room": room}

# 建立包含多工具與系統事實 (System Facts) 的 Agent
multi_agent = needle.Needle(
    tools=[control_lights, set_thermostat, start_robot_vacuum],
    system="date: 2026-09-20 Sun 08:00; device: smart_home_hub; location: living_room"
)

# 執行包含三個硬體動作的複合語句
complex_query = "把客廳燈調暗到 35% 暖白光，將空調溫度設為 23 度，並派遣掃地機器人去清掃廚房。"
multi_res = multi_agent.run(complex_query)

print(f"🎯 置信度: {multi_res['confidence']}")
print(f"🧠 推理依據: {multi_res['reasoning']}")
print("\n🛠️ 生成的工具調用序列:")
for idx, call in enumerate(multi_res["function_calls"], 1):
    print(f"  {idx}. [{call['name']}] -> 參數: {call['arguments']}")

print("\n⚡ 實際執行的設備回傳:")
for res in multi_res["results"]:
    print(f"  ✓ {res}")

--- 
## 步驟 4：零幻覺與安全防護機制 (Anti-Hallucination & Refusal)

在物聯網與硬體控制中，最危險的事情就是**模型產生幻覺、擅自執行未定義的指令**。

如果使用者輸入與註冊工具無關的提問（例如閒聊、哲學問題或未開通的功能），Needle 會如何反應？

In [ ]:
# 提問一個超出工具定義範圍的問題
off_topic_query = "誰寫了哈姆雷特與羅密歐與茱麗葉？"
refusal_res = multi_agent.run(off_topic_query)

print("當面對無關提問時的模型輸出：")
print(f"- 呼叫清單 function_calls: {refusal_res['function_calls']}")
print(f"- 執行結果 results: {refusal_res['results']}")
print(f"- 置信度 confidence: {refusal_res['confidence']}")

if not refusal_res["function_calls"]:
    print("\n🛡️ 安全攔截成功：Needle 主動返回空列表拒絕 (Safe Refusal)，避免亂點設備！")

--- 
## 步驟 5：置信度分級決策管線 (Confidence Gating Pipeline)

Needle 的置信度分數經過校準（Calibrated）。在嚴格的安全系統中，建議採用三級決策邏輯：

In [ ]:
def dispatch_request(agent, user_text, threshold=0.75):
    turn = agent.complete(user_text)
    calls = turn.get("function_calls", [])
    suppressed = turn.get("suppressed_calls", [])
    conf = turn.get("confidence")
    
    print(f"\n[輸入]: \"{user_text}\"")
    print(f"[評分]: 置信度 = {conf}")
    
    # 分級決策
    if calls and (conf is not None and conf >= threshold):
        print(f"🟢 【動作：直接執行】置信度高於 {threshold}，立即向設備發出控制信號。")
        return agent.run(user_text)["results"]
    elif calls or suppressed:
        candidate = calls or suppressed
        print(f"🟡 【動作：二次確認】置信度較低或屬於疑似呼叫，在螢幕提示用戶確認: {candidate}")
        return "WAIT_USER_CONFIRMATION"
    else:
        print("🔴 【動作：安全拒絕】查無匹配功能或無法支援，語音提示用戶無法處理。")
        return "REFUSED"

# 測試三種不同情境
dispatch_request(multi_agent, "把臥室的燈關掉")
dispatch_request(multi_agent, "把那個東西弄一下好嗎")
dispatch_request(multi_agent, "今天股票行情如何")

--- 
## 步驟 6：離線強型別結構化萃取 (Pydantic Extraction)

除了控制工具，Needle 還能將雜亂、非結構化的文本直接提取為 Pydantic 物件。
這在端側離線掃描發票、會議通知、聯絡人資訊等任務上極度好用！

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# 定義會議預約的資料模型
class MeetingSchedule(BaseModel):
    topic: str = Field(description="會議主題")
    time_str: str = Field(description="會議時間")
    room: Literal["Meeting Room A", "Meeting Room B", "Online Zoom"]
    participants: List[str] = Field(description="與會人員名單")
    need_projector: bool = Field(default=False, description="是否需要投影機")

# 雜亂的語音轉文字訊息
noisy_message = """
嗨各位，下週三上午十點半我們需要在 Meeting Room A 開一下 Q4 產品架構審查會，
請 Wayne、Alice 跟 Bob 一定要到，記得先借投影機喔！
"""

# 一行萃取強型別 Pydantic 實例
meeting = needle.extract(noisy_message, schema=MeetingSchedule)

print("=== 結構化提取結果 ===")
print(meeting)
print("\n欄位型別驗證:")
print(f"- 主題: {meeting.topic} (型別: {type(meeting.topic).__name__})")
print(f"- 會議室: {meeting.room}")
print(f"- 與會者: {meeting.participants} (列表個數: {len(meeting.participants)})")
print(f"- 需要投影機: {meeting.need_projector}")

--- 
## 步驟 7：體驗內建的預置環境 (Ready-Made Environments)

Needle 內建了 6 大常見領域的現成環境：
- `smart_home` (智慧家居)
- `media_player` (媒體播放器)
- `wearable` (智慧手錶穿戴)
- `productivity` (個人生產力/日曆)
- `kitchen_appliance` (智慧廚房家電)
- `data_capture` (表單數據採集)

每個環境都自帶標準工具集、系統事實、以及凍結的驗收測試案例 (Acceptance Suite)：

In [ ]:
from needle.environments import smart_home

# 使用內建的 smart_home agent 進行推論
res = smart_home.agent.complete("dim the study lights to 30 percent")
print("smart_home 環境推論輸出:")
print(json.dumps(res, indent=2))

# 執行環境內建的測試案例套件
print("\n執行 smart_home 驗收測試:")
test_summary = smart_home.run_tests()
print(f"測試通過率結果: {test_summary}")

--- 
## 步驟 8：進階微調 (Fine-Tuning) 與多平臺導出

Needle 採用階梯式架構 (Laddered Architecture)，能輕鬆使用 LoRA 進行微調，並切換成 2~20 層的不同深度。

可以在終端機中透過 CLI 執行：

```bash
# 1. 安裝訓練依賴 (包含 JAX, Flax, Optax)
pip install "cactus-needle[train]"

# 2. 使用 JSONL 數據微調 LoRA 適配器
needle finetune data.jsonl --epochs 3 --out adapter.safetensors

# 3. 合併並切出 8 層 (52M) 的輕量化模型，導出為 .cact 格式
needle build --lora adapter.safetensors --layers 8 --out my_model.cact

# 4. 打包特定平臺的原生引擎 (例如樹莓派 ARM64)
needle build --platform linux-arm64 --layers 8 --out ./pi_device
```

--- 
## 總結與學習資源

恭喜您完成了 Needle 3 的互動式教學！我們回顧了：
1. 如何以純 Python 裝飾器 `@needle.tool` 快速註冊硬體工具。
2. 多工具複合意圖的精確執行流程。
3. 藉由約束解碼保證 100% 正確的 JSON 輸出，以及超出範圍時的主動安全拒絕。
4. 結合 Pydantic 實現強型別的端側結構化資料萃取。
5. 運用內建環境加速原型開發。

### 🔗 相關資源：
- 完整圖文技術分析報告：請見倉庫根目錄的 `report.html`
- GitHub 倉庫：[Child-pi/needle](https://github.com/Child-pi/needle)
- 模型權重與跨平臺二進制檔：[Hugging Face Cactus-Compute/needle3](https://huggingface.co/Cactus-Compute/needle3)
- 官方互動 Playground：[cactuscompute.com/needle](https://cactuscompute.com/needle)